# 02 · Modeling

**Goal:** fit an interpretable GLM and a gradient boosting model on 2022–2023, ready to compare on 2024.

1. Load training data
2. GLM: binomial with cloglog link and log(exposure) offset
3. Interpret GLM coefficients (relativities)
4. LightGBM benchmark
5. Notes on modeling choices

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config

from src.train import fit_glm, fit_lgbm

In [ ]:
train = pd.read_csv(config.TRAIN_FILE)
print(train.shape, f"claim rate = {train[config.TARGET].mean():.2%}")

## 2. GLM

Why the **cloglog link with a log(exposure) offset**? If claims arrive as a Poisson process with rate λ, the probability of at least one claim over exposure *t* is 1 − exp(−λt). Taking cloglog of that gives log(λ) + log(t), so log(exposure) enters as an offset. TODO: explain this in your own words in the report.

In [ ]:
glm = fit_glm(train)
glm.summary()

## 3. Relativities

exp(β) is the multiplicative effect on the claim rate. Example: 1.25 means 25% higher risk than the reference level, all else equal.

In [ ]:
relativities = pd.DataFrame({
    "relativity": np.exp(glm.params),
    "ci_low": np.exp(glm.conf_int()[0]),
    "ci_high": np.exp(glm.conf_int()[1]),
    "p_value": glm.pvalues,
}).round(3)
relativities

**Interpretation:**

TODO: which factors matter most? Any surprises?

## 4. LightGBM benchmark

In [ ]:
lgbm = fit_lgbm(train)

## 5. Modeling choices

TODO: write down
- why you did not oversample or reweight the minority class
- how you tuned LightGBM (and that tuning used only training years)
- anything you tried that didn't work